# 11 — SRNet-v11 CPU recovery and secondary robustness analysis

This notebook is a **detector-development recovery run** after Patch 10 failed its pre-test sanity gate (`AUC ≈ 0.508`). The failed Patch-10 detector never scored TEST.

The RDH method remains frozen. This notebook may change **only the independent SRNet-architecture detector training protocol** using TRAIN/development data. It must not change the allocator, `alpha=0.25`, payload definitions, teacher, frozen TEST source IDs, or notebook-06 results.

Changes relative to Patch 10:

1. use 5,250 TRAIN source images for detector fitting and 750 TRAIN images for development;
2. use a real minibatch of 4 cover–stego pairs (8 images) so BatchNorm sees a larger batch;
3. add deterministic curriculum warm-up at 0.012 net bpp before target-payload training at 0.009 net bpp;
4. sample 3,000 source pairs per epoch from the larger fitting pool;
5. select the best checkpoint **only within the final low-learning-rate target stage**, using development AUC;
6. save resumable epoch checkpoints and print within-epoch progress;
7. keep TEST closed unless the frozen pre-test gate `AUC >= 0.65` is passed.

This is still a CPU-constrained SRNet-architecture experiment, not a reproduction of the original 500k-iteration SRNet training schedule.

In [ ]:
from pathlib import Path
import gc, hashlib, json, os, time, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# CPU configuration: safe on CPU-only Docker; does not alter the scientific protocol.
CPU_THREADS=max(1,min(8,os.cpu_count() or 1))
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.backends.mkldnn.enabled=True

from rdhlab.io import read_gray
from rdhlab.blockcodec import analyze_blocks
from rdhlab.pipeline import prepare_image_context, run_frozen_image_precomputed
from rdhlab.detectors import detector_metrics, paired_detector_bootstrap
from rdhlab.transfer import paired_method_bootstrap
from rdhlab.freeze_protocol import sha256_file, stable_id_hash
from rdhlab.srnet_secondary_v11 import (
    train_srnet_v11, score_srnet, save_srnet, load_srnet,
    minimal_detection_error, paired_pe_bootstrap, parameter_count,
)

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=max(5000,int(config['statistics'].get('bootstrap_resamples',5000)))

manifest=pd.read_csv(config['dataset']['prepared_manifest'])
train=manifest[manifest.split=='train'].reset_index(drop=True)
test=manifest[manifest.split=='test'].reset_index(drop=True)
assert len(train)==6000 and len(test)==2000
assert train.source_id.astype(str).is_unique
assert test.source_id.astype(str).is_unique
assert set(train.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))

allocator_path=Path('/workspace/config/frozen_allocator.json')
allocator=json.loads(allocator_path.read_text())
alpha=float(allocator['alpha'])
primary_bpp=float(allocator['teacher_payload_bpp'])
bs=int(allocator.get('block_size',config['dataset']['block_size']))
assert np.isclose(alpha,0.25)
assert np.isclose(primary_bpp,0.009)

out06=Path('/workspace/results/frozen_test_final')
complete=json.loads((out06/'test_run_complete.json').read_text())
protocol06=json.loads((out06/'test_protocol.json').read_text())
common=pd.read_csv(out06/'common_feasible_ids.csv')
common['source_id']=common.source_id.astype(str)
per06=pd.read_csv(out06/'per_image.csv')
per06['source_id']=per06.source_id.astype(str)

risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
if sha256_file(risk_path)!=allocator['provenance_sha256']['srm_teacher_local_risk_joblib']:
    raise RuntimeError('Frozen SRM-derived local-risk model hash mismatch.')
if stable_id_hash(test.source_id.astype(str).tolist())!=complete['test_source_ids_sha256']:
    raise RuntimeError('Frozen test source-ID hash mismatch.')
local_risk=joblib.load(risk_path)

context_dir=out06/'contexts'/protocol06['context_tag']
if not context_dir.exists():
    raise RuntimeError(f'Frozen context cache missing: {context_dir}')

OUT=Path('/workspace/results/srnet_secondary_v11')
CACHE=OUT/'cache'
CKPT=OUT/'checkpoints'
MODEL_DIR=Path('/workspace/results/models')
for p in (OUT,CACHE,CKPT,MODEL_DIR):
    p.mkdir(parents=True,exist_ok=True)

print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
print('CPU threads:',torch.get_num_threads())
print('Frozen alpha:',alpha)
print('Primary payload:',primary_bpp)
print('Block size:',bs)
print('Train/test:',len(train),len(test))

## 11.1 Freeze the revised detector-development protocol

Patch 10 demonstrated that the short 2-pair-microbatch run learned the fitting data but failed to generalize on the development partition. The revised protocol below is fixed before the new model is trained. TEST remains inaccessible to detector fitting and checkpoint selection.

In [ ]:
FIT_START=0
FIT_N=5250
DEV_START=5250
DEV_N=750
CURRICULUM_BPP=0.012
TARGET_BPP=0.009
CURRICULUM_EPOCHS=2
TARGET_EPOCHS_STAGE1=7
TARGET_EPOCHS_STAGE2=4
PAIR_BATCH=4
PAIRS_PER_EPOCH=3000
LR_STAGE1=1e-3
LR_STAGE2=1e-4
WEIGHT_DECAY=2e-4
SANITY_AUC_THRESHOLD=0.65

srnet_fit=train.iloc[FIT_START:FIT_START+FIT_N].reset_index(drop=True)
srnet_dev=train.iloc[DEV_START:DEV_START+DEV_N].reset_index(drop=True)
assert len(srnet_fit)==FIT_N and len(srnet_dev)==DEV_N
assert set(srnet_fit.source_id.astype(str)).isdisjoint(set(srnet_dev.source_id.astype(str)))
assert set(srnet_fit.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))
assert set(srnet_dev.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))

v10_sanity_path=Path('/workspace/results/srnet_secondary/srnet_dev_sanity.json')
v10_sanity=json.loads(v10_sanity_path.read_text()) if v10_sanity_path.exists() else None

protocol={
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS_DETECTOR_DEVELOPMENT_V11',
    'target_journal':'Signal Processing',
    'detector':'SRNet architecture (Boroumand, Chen, Fridrich 2019)',
    'role':'second independently trained neural steganalyzer; architectural/model independence, not training-source independence',
    'teacher':'SRM-derived local-risk teacher',
    'primary_confirmatory_detector':'enhanced residual CNN from 05e/07',
    'allocator_alpha_frozen':alpha,
    'primary_payload_bpp':TARGET_BPP,
    'curriculum_payload_bpp':CURRICULUM_BPP,
    'comparison':['joint','predictability'],
    'training_allocation':'random',
    'fit_start':FIT_START,'fit_n':FIT_N,
    'dev_start':DEV_START,'dev_n':DEV_N,
    'curriculum_epochs':CURRICULUM_EPOCHS,
    'target_epochs_stage1':TARGET_EPOCHS_STAGE1,
    'target_epochs_stage2':TARGET_EPOCHS_STAGE2,
    'pair_batch_size':PAIR_BATCH,
    'images_per_training_batch':2*PAIR_BATCH,
    'pairs_sampled_per_epoch':PAIRS_PER_EPOCH,
    'gradient_accumulation':False,
    'checkpoint_selection':'maximum target-payload development AUC within final low-learning-rate stage only',
    'lr_stage1':LR_STAGE1,'lr_stage2':LR_STAGE2,
    'weight_decay':WEIGHT_DECAY,
    'sanity_auc_threshold':SANITY_AUC_THRESHOLD,
    'fixed_fpr':fixed_fpr,
    'bootstrap_resamples':n_boot,
    'test_results_previously_known_from_primary_experiment':True,
    'srnet_test_must_not_be_scored_before_gate':True,
    'no_allocator_or_payload_retuning_permitted':True,
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'test_source_ids_sha256':complete['test_source_ids_sha256'],
    'fit_ids_sha256':stable_id_hash(srnet_fit.source_id.astype(str).tolist()),
    'dev_ids_sha256':stable_id_hash(srnet_dev.source_id.astype(str).tolist()),
    'patch10_failed_pretest_attempt':v10_sanity,
    'published_srnet_training_not_reproduced':True,
}
protocol_path=OUT/'srnet_v11_protocol_pretest.json'
protocol_path.write_text(json.dumps(protocol,indent=2),encoding='utf-8')
protocol_sha=sha256_file(protocol_path)
print('PRE-TEST PROTOCOL SHA256:',protocol_sha)
print(json.dumps(protocol,indent=2))

## 11.2 Build or reload TRAIN/development caches

Two TRAIN-only fitting sets are generated with the same frozen reversible codec and deterministic random allocation: a 0.012-bpp curriculum set and the 0.009-bpp target set. The 750-image development partition is generated only at 0.009 bpp. Arrays are cached as `.npy` files so a kernel restart does not require regenerating all pairs.

In [ ]:
def random_order_for_plans(plans,source_id,payload):
    bids=np.asarray([p.block_id for p in plans],dtype=int)
    digest=hashlib.sha256(f'{seed}|srnet-v11|{payload:.6f}|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    out=bids.copy(); rng.shuffle(out)
    return out

def cache_key(label,payload):
    return f'{label}_{payload:.3f}'.replace('.','p')

def load_or_build_pairs(frame,label,payload):
    key=cache_key(label,payload)
    cp=CACHE/f'{key}_covers.npy'
    sp=CACHE/f'{key}_stegos.npy'
    ip=CACHE/f'{key}_ids.csv'
    mp=CACHE/f'{key}_meta.json'
    expected_hash=stable_id_hash(frame.source_id.astype(str).tolist())
    if cp.exists() and sp.exists() and ip.exists() and mp.exists():
        meta=json.loads(mp.read_text())
        if (meta.get('source_ids_sha256')==expected_hash and
            np.isclose(float(meta.get('payload_bpp',-1)),float(payload)) and
            int(meta.get('block_size',-1))==bs):
            covers=np.load(cp,mmap_mode='r')
            stegos=np.load(sp,mmap_mode='r')
            ids=pd.read_csv(ip).source_id.astype(str).tolist()
            if len(covers)==len(stegos)==len(ids):
                print('CACHE HIT:',key,'pairs=',len(ids))
                return covers,stegos,ids
        raise RuntimeError(f'Stale/incompatible cache exists for {key}; remove only that cache after inspection.')

    covers=[]; stegos=[]; ids=[]; skipped=[]
    for j,row in frame.iterrows():
        sid=str(row.source_id)
        x=read_gray(row.path)
        plans=analyze_blocks(x,bs)
        order=random_order_for_plans(plans,sid,payload)
        rr=run_frozen_image_precomputed(
            x,sid,float(payload),'random',{'random':order},[],bs,seed,False,None,plans=plans
        )
        if rr['feasible']:
            if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
                raise RuntimeError(f'Reversibility invariant failed for {label}: {sid}')
            covers.append(np.asarray(x,dtype=np.uint8))
            stegos.append(np.asarray(rr['stego'],dtype=np.uint8))
            ids.append(sid)
        else:
            skipped.append(sid)
        if (j+1)%250==0 or j+1==len(frame):
            print(label,payload,j+1,'/',len(frame),'feasible',len(covers),flush=True)

    covers=np.stack(covers)
    stegos=np.stack(stegos)
    np.save(cp,covers,allow_pickle=False)
    np.save(sp,stegos,allow_pickle=False)
    pd.DataFrame({'source_id':ids}).to_csv(ip,index=False)
    meta={
        'label':label,'payload_bpp':float(payload),'block_size':bs,
        'requested_sources':len(frame),'feasible_pairs':len(ids),'skipped':skipped,
        'source_ids_sha256':expected_hash,'protocol_sha256':protocol_sha,
    }
    mp.write_text(json.dumps(meta,indent=2),encoding='utf-8')
    del covers,stegos
    gc.collect()
    return np.load(cp,mmap_mode='r'),np.load(sp,mmap_mode='r'),ids

fit_c12,fit_s12,fit_ids12=load_or_build_pairs(srnet_fit,'fit',CURRICULUM_BPP)
fit_c09,fit_s09,fit_ids09=load_or_build_pairs(srnet_fit,'fit',TARGET_BPP)
dev_c09,dev_s09,dev_ids09=load_or_build_pairs(srnet_dev,'dev',TARGET_BPP)

print('Curriculum fit feasibility:',len(fit_ids12),'/',len(srnet_fit))
print('Target fit feasibility:',len(fit_ids09),'/',len(srnet_fit))
print('Target dev feasibility:',len(dev_ids09),'/',len(srnet_dev))
if len(fit_ids12)<0.90*len(srnet_fit) or len(fit_ids09)<0.90*len(srnet_fit) or len(dev_ids09)<0.90*len(srnet_dev):
    raise RuntimeError('Unexpectedly low detector-development feasibility; inspect before training.')

## 11.3 Train/resume SRNet-v11 on TRAIN only

The training function writes `srnet_v11_last.pt` after every epoch. Re-running this cell resumes from the last completed epoch if the pre-test protocol hash is unchanged. The best checkpoint is selected only during the final low-learning-rate target stage.

In [ ]:
srnet,history,device,selection=train_srnet_v11(
    curriculum_covers=fit_c12,curriculum_stegos=fit_s12,
    target_covers=fit_c09,target_stegos=fit_s09,
    validation=(dev_c09,dev_s09),
    curriculum_epochs=CURRICULUM_EPOCHS,
    target_epochs_stage1=TARGET_EPOCHS_STAGE1,
    target_epochs_stage2=TARGET_EPOCHS_STAGE2,
    pair_batch_size=PAIR_BATCH,
    pairs_per_epoch=PAIRS_PER_EPOCH,
    lr_stage1=LR_STAGE1,lr_stage2=LR_STAGE2,
    weight_decay=WEIGHT_DECAY,
    seed=seed+11000,fixed_fpr=fixed_fpr,
    checkpoint_dir=CKPT,protocol_sha256=protocol_sha,
    resume=True,log_interval=100,
)

hist=pd.DataFrame(history)
hist.to_csv(OUT/'srnet_v11_train_history.csv',index=False)
display(hist)
print('Device:',device)
print('Trainable parameter count:',parameter_count(srnet))
print('Selected checkpoint:',selection)

model_path=MODEL_DIR/'srnet_secondary_v11.pt'
save_srnet(srnet,model_path,{
    'analysis_status':'post-hoc secondary robustness detector v11',
    'architecture':'SRNet topology, Boroumand et al. 2019',
    'target_payload_bpp':TARGET_BPP,'curriculum_payload_bpp':CURRICULUM_BPP,
    'training_allocation':'random','train_split_only':True,
    'fit_pool_pairs_target':len(fit_c09),'fit_pool_pairs_curriculum':len(fit_c12),
    'dev_pairs':len(dev_c09),'pair_batch_size':PAIR_BATCH,
    'pairs_sampled_per_epoch':PAIRS_PER_EPOCH,
    'curriculum_epochs':CURRICULUM_EPOCHS,
    'target_epochs_stage1':TARGET_EPOCHS_STAGE1,
    'target_epochs_stage2':TARGET_EPOCHS_STAGE2,
    'selected_epoch':selection['best_epoch'],'selected_dev_auc':selection['best_auc'],
    'selection_rule':'best development AUC in final low-LR target stage',
    'protocol_sha256':protocol_sha,'seed':seed+11000,
})

## 11.4 Pre-test sanity gate

Only the selected checkpoint is evaluated on the target-payload development partition. If `AUC < 0.65`, the notebook stops here. In that case **TEST remains unscored** and no RDH parameter may be changed.

In [ ]:
dev_cover_scores=score_srnet(srnet,dev_c09,device=device,batch_size=max(4,PAIR_BATCH))
dev_stego_scores=score_srnet(srnet,dev_s09,device=device,batch_size=max(4,PAIR_BATCH))
y=np.tile([0,1],len(dev_cover_scores))
sc=np.column_stack([dev_cover_scores,dev_stego_scores]).reshape(-1)
met=detector_metrics(y,sc,fixed_fpr)
ci=paired_detector_bootstrap(
    dev_cover_scores,dev_stego_scores,fixed_fpr=fixed_fpr,
    n_resamples=max(n_boot,3000),confidence=confidence,seed=seed+11100,
)
dev_pe=minimal_detection_error(dev_cover_scores,dev_stego_scores)

sanity={
    'sanity_pass':bool(met['auc']>=SANITY_AUC_THRESHOLD),
    'sanity_auc_threshold':SANITY_AUC_THRESHOLD,
    'pairs':len(dev_cover_scores),
    'auc':float(met['auc']),
    'auc_ci_low':float(ci['auc_low']),'auc_ci_high':float(ci['auc_high']),
    'tpr_at_5pct_fpr':float(met['tpr_at_fpr']),
    'tpr_ci_low':float(ci['tpr_low']),'tpr_ci_high':float(ci['tpr_high']),
    'pe':float(dev_pe),
    'selected_epoch':selection['best_epoch'],
    'model_sha256':sha256_file(model_path),
    'protocol_sha256':protocol_sha,
    'test_split_scored':False,
}
(OUT/'srnet_v11_dev_sanity.json').write_text(json.dumps(sanity,indent=2),encoding='utf-8')
print(json.dumps(sanity,indent=2))

if not sanity['sanity_pass']:
    raise RuntimeError(
        f"SRNet-v11 failed the pre-test gate: AUC={sanity['auc']:.3f} < {SANITY_AUC_THRESHOLD:.2f}. "
        'TEST was not scored. Preserve this attempt; do not modify the frozen RDH allocator.'
    )
print('SANITY PASSED. Freeze SRNet-v11 before any TEST scoring.')

In [ ]:
lock={
    **protocol,
    'protocol_sha256':protocol_sha,
    'srnet_model_sha256':sha256_file(model_path),
    'srnet_model_metadata_sha256':sha256_file(model_path.with_suffix(model_path.suffix+'.json')),
    'dev_sanity_auc':sanity['auc'],'dev_sanity_pass':sanity['sanity_pass'],
    'selected_epoch':selection['best_epoch'],
    'protocol_locked_before_srnet_test_scoring':True,
}
(OUT/'srnet_v11_pretest_lock.json').write_text(json.dumps(lock,indent=2),encoding='utf-8')
print('SRNet-v11 pre-test lock SHA256:',sha256_file(OUT/'srnet_v11_pretest_lock.json'))

## 11.5 Frozen TEST scoring — executes only after the gate passes

Only the already frozen primary payload and common-feasible TEST source IDs are used. Each regenerated stego image is checked against notebook-06 reversibility, net-payload, and PSNR invariants before SRNet-v11 scoring.

In [ ]:
srnet,device=load_srnet(model_path)
primary_ids=common[np.isclose(common.target_net_bpp.astype(float),TARGET_BPP)].source_id.astype(str).tolist()
if len(primary_ids)==0:
    raise RuntimeError('No common-feasible primary-payload IDs.')

test_by_id={str(r.source_id):r for _,r in test.iterrows()}
missing=[sid for sid in primary_ids if sid not in test_by_id]
if missing:
    raise RuntimeError(f'{len(missing)} frozen IDs missing from test manifest')

def load_context(x,sid):
    p=context_dir/f'{sid}.joblib'
    if p.exists():
        ctx=joblib.load(p)
        if ctx.get('context_tag')!=protocol06['context_tag']:
            raise RuntimeError(f'Stale context for {sid}')
        return ctx['orders'],ctx['block_rows'],ctx['plans']
    return prepare_image_context(x,sid,local_risk,alpha,bs,seed)

cover_images=[read_gray(test_by_id[sid].path) for sid in primary_ids]
cover_scores=score_srnet(srnet,cover_images,device=device,batch_size=8)
cover_score_map=dict(zip(primary_ids,map(float,cover_scores)))

def flush_score_batch(model,device,batch_imgs,batch_meta,rows):
    if not batch_imgs:
        return
    ss=score_srnet(model,batch_imgs,device=device,batch_size=8)
    for meta,score in zip(batch_meta,ss):
        cs=cover_score_map[meta['source_id']]
        rows.append({**meta,'cover_score':cs,'stego_score':float(score),'score_delta':float(score-cs)})

rows=[]
for strategy in ['predictability','joint']:
    batch_imgs=[]; batch_meta=[]
    for k,sid in enumerate(primary_ids,1):
        x=read_gray(test_by_id[sid].path)
        orders,br,plans=load_context(x,sid)
        rr=run_frozen_image_precomputed(
            x,sid,TARGET_BPP,strategy,orders,br,bs,seed,False,None,plans=plans
        )
        if not rr['feasible']:
            raise RuntimeError(f'Frozen common-feasible case became infeasible: {sid} {strategy}')
        if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
            raise RuntimeError(f'Reversibility invariant failed: {sid} {strategy}')
        old=per06[
            (per06.source_id==sid) & (per06.strategy.astype(str)==strategy) &
            np.isclose(per06.target_net_bpp.astype(float),TARGET_BPP)
        ]
        if len(old)!=1:
            raise RuntimeError(f'Frozen case lookup failed: {sid} {strategy}')
        old=old.iloc[0]
        if int(rr['net_payload_bits'])!=int(old['net_payload_bits']):
            raise RuntimeError(f'Net-payload mismatch vs frozen 06: {sid} {strategy}')
        if not np.isclose(float(rr['psnr']),float(old['psnr']),rtol=0,atol=1e-10):
            raise RuntimeError(f'PSNR mismatch vs frozen 06: {sid} {strategy}')
        batch_imgs.append(rr['stego'])
        batch_meta.append({'source_id':sid,'strategy':strategy,'target_net_bpp':TARGET_BPP})
        if len(batch_imgs)>=8:
            flush_score_batch(srnet,device,batch_imgs,batch_meta,rows)
            batch_imgs=[]; batch_meta=[]
        if k%250==0 or k==len(primary_ids):
            print(strategy,k,'/',len(primary_ids),flush=True)
    flush_score_batch(srnet,device,batch_imgs,batch_meta,rows)

scores=pd.DataFrame(rows)
scores.to_csv(OUT/'srnet_v11_primary_scores.csv',index=False)
print('Saved rows:',len(scores))

In [ ]:
summary_rows=[]
for strategy,g in scores.groupby('strategy',sort=False):
    c0=g.cover_score.to_numpy(float); s0=g.stego_score.to_numpy(float)
    y=np.tile([0,1],len(g)); sc=np.column_stack([c0,s0]).reshape(-1)
    m=detector_metrics(y,sc,fixed_fpr)
    ci=paired_detector_bootstrap(
        c0,s0,fixed_fpr=fixed_fpr,n_resamples=n_boot,
        confidence=confidence,seed=seed+11200+(0 if strategy=='predictability' else 1),
    )
    summary_rows.append({
        'strategy':strategy,'target_net_bpp':TARGET_BPP,'pairs':len(g),
        'auc':float(m['auc']),'auc_ci_low':float(ci['auc_low']),'auc_ci_high':float(ci['auc_high']),
        'tpr_at_5pct_fpr':float(m['tpr_at_fpr']),
        'tpr_ci_low':float(ci['tpr_low']),'tpr_ci_high':float(ci['tpr_high']),
        'pe':float(minimal_detection_error(c0,s0)),
        'score_delta_mean':float(np.mean(s0-c0)),
    })
summary=pd.DataFrame(summary_rows)
summary.to_csv(OUT/'srnet_v11_primary_summary.csv',index=False)
display(summary)

pred=scores[scores.strategy=='predictability'].set_index('source_id').loc[primary_ids]
joint=scores[scores.strategy=='joint'].set_index('source_id').loc[primary_ids]
c0=joint.cover_score.to_numpy(float)
if not np.allclose(c0,pred.cover_score.to_numpy(float),rtol=0,atol=0):
    raise RuntimeError('Cover-score alignment failure.')

paired=paired_method_bootstrap(
    c0,joint.stego_score.to_numpy(float),pred.stego_score.to_numpy(float),
    fixed_fpr=fixed_fpr,n_resamples=n_boot,confidence=confidence,seed=seed+11300,
)
pe_boot=paired_pe_bootstrap(
    c0,joint.stego_score.to_numpy(float),pred.stego_score.to_numpy(float),
    n_resamples=n_boot,confidence=confidence,seed=seed+11310,
)
paired.update(pe_boot)
paired.update({
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS',
    'detector_version':'SRNet-v11','method':'joint','reference':'predictability',
    'target_net_bpp':TARGET_BPP,'alpha':alpha,'pairs':len(primary_ids),
    'srnet_model_sha256':sha256_file(model_path),
    'allocator_sha256':sha256_file(allocator_path),'risk_model_sha256':sha256_file(risk_path),
    'no_retuning_permitted':True,
})
(OUT/'srnet_v11_joint_vs_predictability.json').write_text(json.dumps(paired,indent=2),encoding='utf-8')
print(json.dumps(paired,indent=2))

In [ ]:
fig,ax=plt.subplots(figsize=(6.2,4.2))
z=summary.set_index('strategy').loc[['predictability','joint']]
ax.bar(['Predictability','Joint'],z.auc.values)
ax.axhline(0.5,linewidth=1)
ax.set_ylabel('SRNet-v11 ROC-AUC')
ax.set_title('Secondary robustness analysis at 0.009 net bpp')
fig.tight_layout()
fig.savefig(OUT/'srnet_v11_primary_auc.png',dpi=300)
fig.savefig(OUT/'srnet_v11_primary_auc.svg')
plt.show()

final={
    'status':'COMPLETE','analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS',
    'detector_version':'SRNet-v11','primary_payload_bpp':TARGET_BPP,'alpha':alpha,
    'pairs':len(primary_ids),'joint_auc':float(z.loc['joint','auc']),
    'predictability_auc':float(z.loc['predictability','auc']),
    'joint_tpr5':float(z.loc['joint','tpr_at_5pct_fpr']),
    'predictability_tpr5':float(z.loc['predictability','tpr_at_5pct_fpr']),
    'joint_pe':float(z.loc['joint','pe']),'predictability_pe':float(z.loc['predictability','pe']),
    'auc_diff_joint_minus_predictability':float(paired['auc_diff']),
    'auc_diff_ci_low':float(paired['auc_diff_low']),'auc_diff_ci_high':float(paired['auc_diff_high']),
    'tpr_diff_joint_minus_predictability':float(paired['tpr_diff']),
    'tpr_diff_ci_low':float(paired['tpr_diff_low']),'tpr_diff_ci_high':float(paired['tpr_diff_high']),
    'pe_diff_joint_minus_predictability':float(paired['pe_diff']),
    'pe_diff_ci_low':float(paired['pe_diff_low']),'pe_diff_ci_high':float(paired['pe_diff_high']),
    'interpretation_rule':'supportive if joint has lower AUC/TPR and higher Pe; report observed result regardless of direction',
    'allocator_retuned':False,'test_split_scored':True,
}
(OUT/'srnet_v11_secondary_summary.json').write_text(json.dumps(final,indent=2),encoding='utf-8')
print(json.dumps(final,indent=2))

## Outputs to preserve

If the sanity gate passes and TEST scoring completes, preserve:

- `results/srnet_secondary_v11/srnet_v11_protocol_pretest.json`
- `results/srnet_secondary_v11/srnet_v11_train_history.csv`
- `results/srnet_secondary_v11/srnet_v11_dev_sanity.json`
- `results/srnet_secondary_v11/srnet_v11_pretest_lock.json`
- `results/srnet_secondary_v11/srnet_v11_primary_summary.csv`
- `results/srnet_secondary_v11/srnet_v11_joint_vs_predictability.json`
- `results/srnet_secondary_v11/srnet_v11_secondary_summary.json`

If the gate fails, preserve the protocol, history, sanity JSON, and checkpoints as a failed pre-test detector-development attempt. Do not score TEST.